In [4]:
import regex as re
import os
import torch
import sys

sys.path.append("harpak_lab/cs336/assignment1-basics/cs336_basics")
from collections import Counter
from multiprocessing import Pool
from typing import BinaryIO, List, Tuple, Dict
from cs336_basics.transformer import *

In [10]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"

model = TransformerLM(
    vocab_size=10000,
    context_length=256,
    num_layers=4,
    d_model=512,
    num_heads=16,
    d_ff=1344,
    theta=10000.0,
).to(device)

optimizer = AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
)

iteration = load_checkpoint(
    src="../checkpoints/model_final_iter40000",
    model=model,
    optimizer=optimizer,
)

model.eval()

text = decoding(
    model=model,
    prompt="Do you know what is taylor expansion",
    max_generate_tokens=100,
    temperature=0.8,
    p_threshold=0.9,
    vocab_filepath="tiny_stories_vocab.json",
    merges_filepath="tiny_stories_merges.json",
)

print(text)

? That is not how you think a spicy plant, but how do you know that?"
"Yes, sir, it's my job. Thank you, sir. I like to look at you and listen to you. You are very kind and smart. You made me happy and proud. I am so happy to see you."
The man smiled and said, "You are welcome, sir. You are very kind and sweet. You made me happy and smart. I am glad you are


In [1]:
import regex as re
import os
import pickle
import time

from collections import Counter
from multiprocessing import Pool
from typing import BinaryIO, List, Tuple, Dict
from functools import partial


def process_chunk(text, special_tokens):
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

    escaped = "|".join(re.escape(t) for t in special_tokens)
    parts = re.split(escaped, text)

    counter = Counter()
    for part in parts:
        for t in re.finditer(PAT, part):
            m = t.group().encode("utf-8")
            counter[tuple(bytes([b]) for b in m)] += 1
            # counter[m] += 1
    return counter


def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))


In [ ]:
f = open("your_file.txt", "rb")

In [6]:
special_tokens = ["<|endoftext|>"]

In [8]:
input_path = '/stor/scratch/Kirkpatrick/xliaoyi/CS336/assignment1-basics/tests/fixtures/tinystories_sample.txt'
chunks = []
with open(input_path, "rb") as f:
    num_processes = 96
    boundaries = find_chunk_boundaries(f, num_processes, special_tokens[0].encode("utf-8"))
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        # print(start, end)
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        chunks.append(chunk)

In [11]:
chunks

["\nOnce upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!\nHe said, “Wow, that is a really amazing vase! Can I buy it?”\nThe shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”\nSo Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was.\nAnd that's how Ben found an amazing vase in the store!\n",
 '<|endoftext|>\nOnce upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.\nOne day, Ollie\'s mom said, "Ollie, hurry and get some fish for dinner!" Ollie swam fast to catch fish. He

In [12]:
with Pool(num_processes) as pool:
    counters = pool.map(
        partial(process_chunk, special_tokens=special_tokens),
        chunks,
    )

In [13]:
all_counter = Counter()
for c in counters:
    all_counter.update(c)

In [14]:
all_counter

Counter({(b'.',): 64,
         (b',',): 39,
         (b' ', b't', b'h', b'e'): 28,
         (b'\n',): 25,
         (b' ', b'a'): 24,
         (b' ', b'a', b'n', b'd'): 22,
         (b' ', b'w', b'a', b's'): 21,
         (b' ', b't', b'o'): 21,
         (b' ', b'w', b'i', b't', b'h'): 12,
         (b' ', b'h', b'i', b's'): 11,
         (b' ', b'L', b'u', b'c', b'y'): 11,
         (b' ', b'i', b't'): 10,
         (b' ', b'd', b'a', b'y'): 9,
         (b' ', b'H', b'e'): 8,
         (b' ', b'"'): 8,
         (b' ', b'B', b'o', b'b'): 8,
         (b' ', b's', b'p', b'i', b'r', b'i', b't'): 8,
         (b' ', b'h', b'e', b'r'): 8,
         (b' ', b'B', b'e', b'n'): 7,
         (b' ', b't', b'h', b'a', b't'): 7,
         (b' ', b's', b'a', b'i', b'd'): 7,
         (b' ', b'f', b'r', b'i', b'e', b'n', b'd', b's'): 7,
         (b' ', b'O', b'l', b'l', b'i', b'e'): 7,
         (b' ', b'T', b'h', b'e', b'y'): 7,
         (b' ', b'p', b'l', b'a', b'y'): 7,
         (b' ', b'b', b'i', b'g'): 7,
  